In [ ]:
pip install pdf2image

Note: you may need to restart the kernel to use updated packages.


In [13]:
from pdf2image import convert_from_path
from pathlib import Path

pdf_path = Path("sourcebook/SymbolSourcebook.pdf")
output_dir = Path("sourcebook/pages")
output_dir.mkdir(parents=True, exist_ok=True)

# Get total page count first
from pdf2image.exceptions import PDFPageCountError
import subprocess
result = subprocess.run(["pdfinfo", str(pdf_path)], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if "Pages:" in line:
        total = int(line.split(":")[1].strip())
        print(f"Total pages: {total}")
        break

# Convert one page at a time so you see progress
for i in range(1, total + 1):
    pages = convert_from_path(
        pdf_path,
        dpi=300,
        fmt="png",
        grayscale=True,
        first_page=i,
        last_page=i
    )
    filename = output_dir / f"page_{i:03d}.png"
    pages[0].save(filename, "PNG")
    print(f"Saved {filename}  ({i}/{total})")

print(f"\nDone — {total} pages exported to {output_dir}")

Total pages: 296
Saved sourcebook/pages/page_001.png  (1/296)
Saved sourcebook/pages/page_002.png  (2/296)
Saved sourcebook/pages/page_003.png  (3/296)
Saved sourcebook/pages/page_004.png  (4/296)
Saved sourcebook/pages/page_005.png  (5/296)
Saved sourcebook/pages/page_006.png  (6/296)
Saved sourcebook/pages/page_007.png  (7/296)
Saved sourcebook/pages/page_008.png  (8/296)
Saved sourcebook/pages/page_009.png  (9/296)
Saved sourcebook/pages/page_010.png  (10/296)
Saved sourcebook/pages/page_011.png  (11/296)
Saved sourcebook/pages/page_012.png  (12/296)
Saved sourcebook/pages/page_013.png  (13/296)
Saved sourcebook/pages/page_014.png  (14/296)
Saved sourcebook/pages/page_015.png  (15/296)
Saved sourcebook/pages/page_016.png  (16/296)
Saved sourcebook/pages/page_017.png  (17/296)
Saved sourcebook/pages/page_018.png  (18/296)
Saved sourcebook/pages/page_019.png  (19/296)
Saved sourcebook/pages/page_020.png  (20/296)
Saved sourcebook/pages/page_021.png  (21/296)
Saved sourcebook/pages/pag

In [ ]:
"""
Event Glyph — Symbol Extractor
Extracts individual symbol crops from the Dreyfuss Symbol Sourcebook.

Folder structure created automatically:
    sourcebook/
    └── dataset/
        ├── crops/          ← extracted symbol PNGs (128x128)
        ├── tagged/         ← crops sorted into category subfolders after tagging
        │   ├── religious_spiritual/
        │   ├── alchemical_occult/
        │   └── ...
        ├── previews/       ← grid preview images for verification
        ├── metadata.json   ← full record of every symbol + tags
        └── log.txt         ← processing log

Usage:
    # Test on one page — saves a preview image
    python grid_extractor.py --preview --input sourcebook/pages/page_054.png

    # Batch all pages
    python grid_extractor.py --batch --input sourcebook/pages/

    # Tag extracted symbols
    python grid_extractor.py --tag
"""

import cv2
import numpy as np
import json
import argparse
import shutil
import sys
from pathlib import Path
from datetime import datetime

try:
    import pytesseract
    OCR_AVAILABLE = True
except ImportError:
    OCR_AVAILABLE = False

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

COLS         = 6
ROWS         = 7
SYMBOL_RATIO = 0.78    # top 78% of cell = symbol, bottom 22% = label text
OUTPUT_SIZE  = 128     # saved crop size in pixels
PADDING      = 8       # whitespace margin inside each crop canvas
LEFT_MARGIN  = 70      # page left margin in pixels
RIGHT_MARGIN = 80      # page right margin in pixels
MIN_INK      = 50000   # minimum ink sum to count as a non-empty cell

OUTPUT_DIR   = Path("sourcebook/dataset")

TAXONOMY = {
    "r": "religious_spiritual",
    "a": "alchemical_occult",
    "s": "scientific_mathematical",
    "n": "nature_cosmological",
    "t": "transport_navigation",
    "m": "medical_health",
    "p": "political_heraldic",
    "c": "cultural_ethnic",
    "g": "geometric_abstract",
    "w": "warning_hazard",
    "i": "industrial_technical",
    "f": "food_drink",
    "d": "delete",
    "u": "unsure",
}


# ─────────────────────────────────────────────
# Folder setup
# ─────────────────────────────────────────────

def setup_folders():
    folders = [
        OUTPUT_DIR / "crops",
        OUTPUT_DIR / "previews",
        OUTPUT_DIR / "tagged",
    ]
    for tag in TAXONOMY.values():
        if tag not in ("delete", "unsure"):
            folders.append(OUTPUT_DIR / "tagged" / tag)
    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)


# ─────────────────────────────────────────────
# Logging
# ─────────────────────────────────────────────

class Logger:
    def __init__(self, log_path):
        self.log_path = log_path
        self._write(f"\n{'='*50}")
        self._write(f"Event Glyph — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        self._write(f"{'='*50}")

    def _write(self, msg):
        print(msg)
        with open(self.log_path, "a") as f:
            f.write(msg + "\n")

    def page_start(self, page_id, page_num, total):
        self._write(f"\n[{page_num}/{total}] {page_id}")

    def page_done(self, page_id, extracted, empty):
        self._write(f"  ✓  {extracted} symbols saved  |  {empty} empty cells ignored")

    def page_skipped(self, page_id, reason):
        self._write(f"  —  skipped: {reason}")

    def page_error(self, page_id, error):
        self._write(f"  ✗  error: {error}")

    def summary(self, processed, total_symbols, skipped, errors):
        self._write(f"\n{'='*50}")
        self._write(f"Batch complete")
        self._write(f"  Pages processed : {processed}")
        self._write(f"  Pages skipped   : {skipped}")
        self._write(f"  Errors          : {errors}")
        self._write(f"  Total symbols   : {total_symbols}")
        self._write(f"  Crops           : {OUTPUT_DIR / 'crops'}")
        self._write(f"  Metadata        : {OUTPUT_DIR / 'metadata.json'}")
        self._write(f"{'='*50}\n")


# ─────────────────────────────────────────────
# Image processing
# ─────────────────────────────────────────────

def load_and_threshold(image_path):
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Could not load: {image_path}")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    thresh = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=31,
        C=10
    )
    return img, gray, thresh


def find_horizontal_grid_lines(gray, line_width_threshold=0.55, min_gap=50):
    """Find thick horizontal lines — the printed row borders of the grid."""
    h, w = gray.shape
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    inv = 255 - binary
    lines = []
    last_y = -min_gap

    for y in range(h):
        row_sum = inv[y, :].sum()
        if row_sum > (line_width_threshold * w * 255):
            if y - last_y > min_gap:
                lines.append(y)
                last_y = y

    return lines


def get_cell_boundaries(img):
    """
    Hybrid grid detection:
    - Horizontal rows from detected printed border lines (adapts per page)
    - Vertical columns from fixed even division (consistent across all pages)
    Falls back to fixed margins if line detection finds fewer than 2 lines.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = img.shape[:2]

    h_lines = find_horizontal_grid_lines(gray)

    if len(h_lines) < 2:
        print("  Warning: grid line detection failed — using fixed margins")
        top    = 210
        bottom = h - 410
        step   = (bottom - top) // ROWS
        h_lines = [top + i * step for i in range(ROWS + 1)]

    usable_w = w - LEFT_MARGIN - RIGHT_MARGIN
    cell_w   = usable_w // COLS
    v_lines  = [LEFT_MARGIN + col * cell_w for col in range(COLS + 1)]

    cells = []
    for row_idx in range(len(h_lines) - 1):
        for col_idx in range(COLS):
            x1 = v_lines[col_idx]
            y1 = h_lines[row_idx]
            x2 = v_lines[col_idx + 1]
            y2 = h_lines[row_idx + 1]
            cells.append((x1, y1, x2, y2, row_idx, col_idx))

    return cells


def is_empty(cell_thresh):
    return (255 - cell_thresh).sum() < MIN_INK


def extract_symbol(cell_gray, split_y):
    """Crop symbol region, find ink bounding box, centre on white canvas."""
    region = cell_gray[:split_y, :]
    _, binary = cv2.threshold(region, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    inv    = 255 - binary
    coords = cv2.findNonZero(inv)

    if coords is None:
        return np.ones((OUTPUT_SIZE, OUTPUT_SIZE), dtype=np.uint8) * 255

    x, y, w, h = cv2.boundingRect(coords)
    cropped = binary[y:y+h, x:x+w]
    inner   = OUTPUT_SIZE - 2 * PADDING
    scale   = min(inner / max(w, 1), inner / max(h, 1))
    new_w   = max(1, int(w * scale))
    new_h   = max(1, int(h * scale))
    resized = cv2.resize(cropped, (new_w, new_h), interpolation=cv2.INTER_AREA)

    canvas = np.ones((OUTPUT_SIZE, OUTPUT_SIZE), dtype=np.uint8) * 255
    ox = (OUTPUT_SIZE - new_w) // 2
    oy = (OUTPUT_SIZE - new_h) // 2
    canvas[oy:oy+new_h, ox:ox+new_w] = resized
    return canvas


def ocr_label(cell_gray, split_y):
    if not OCR_AVAILABLE:
        return ""
    region   = cell_gray[split_y:, :]
    enlarged = cv2.resize(region, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    _, binary = cv2.threshold(enlarged, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    try:
        text = pytesseract.image_to_string(binary, config="--psm 7 --oem 3")
        return text.strip().lower()
    except Exception:
        return ""


# ─────────────────────────────────────────────
# Preview
# ─────────────────────────────────────────────

def save_preview(img, cells, page_id):
    preview = img.copy()
    for (x1, y1, x2, y2, row, col) in cells:
        split_y = y1 + int((y2 - y1) * SYMBOL_RATIO)
        cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 180, 90), 2)
        cv2.line(preview, (x1, split_y), (x2, split_y), (0, 140, 255), 2)
    out = OUTPUT_DIR / "previews" / f"{page_id}_preview.png"
    cv2.imwrite(str(out), preview)
    return out


# ─────────────────────────────────────────────
# Single page extraction
# ─────────────────────────────────────────────

def extract_page(image_path, page_id=None, preview=False, logger=None):
    image_path = Path(image_path)
    page_id    = page_id or image_path.stem
    crops_dir  = OUTPUT_DIR / "crops"

    img, gray, thresh = load_and_threshold(image_path)
    cells = get_cell_boundaries(img)

    if preview:
        out = save_preview(img, cells, page_id)
        msg = f"  Preview saved: {out.resolve()}"
        print(msg) if not logger else logger._write(msg)

    metadata  = []
    extracted = 0
    empty     = 0

    for (x1, y1, x2, y2, row, col) in cells:
        cell_gray   = gray[y1:y2, x1:x2]
        cell_thresh = thresh[y1:y2, x1:x2]

        if cell_gray.size == 0:
            continue

        if is_empty(cell_thresh):
            empty += 1
            continue

        cell_h     = y2 - y1
        split_y    = int(cell_h * SYMBOL_RATIO)
        symbol_img = extract_symbol(cell_gray, split_y)
        label_text = ocr_label(cell_gray, split_y)

        filename = f"{page_id}_r{row:02d}_c{col:02d}.png"
        cv2.imwrite(str(crops_dir / filename), symbol_img)

        metadata.append({
            "file":        filename,
            "page":        page_id,
            "row":         row,
            "col":         col,
            "symbol_name": label_text,
            "tags":        [],
            "notes":       ""
        })
        extracted += 1

    return metadata, extracted, empty


# ─────────────────────────────────────────────
# Batch processing
# ─────────────────────────────────────────────

def batch_extract(input_dir):
    setup_folders()
    input_dir = Path(input_dir)
    log       = Logger(OUTPUT_DIR / "log.txt")
    exts      = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    pages     = sorted(p for p in input_dir.iterdir() if p.suffix.lower() in exts)

    if not pages:
        log._write(f"No images found in {input_dir}")
        return

    log._write(f"Found {len(pages)} pages")
    log._write(f"OCR: {'enabled' if OCR_AVAILABLE else 'disabled'}")

    # Load existing metadata so the batch can be safely resumed
    meta_path    = OUTPUT_DIR / "metadata.json"
    all_meta     = []
    already_done = set()

    if meta_path.exists():
        with open(meta_path) as f:
            all_meta = json.load(f)
        already_done = {r["page"] for r in all_meta}
        log._write(f"Resuming — {len(already_done)} pages already done")

    total_symbols = sum(1 for r in all_meta)
    skipped_pages = 0
    error_pages   = 0
    processed     = 0

    for i, page_path in enumerate(pages, 1):
        page_id = page_path.stem
        log.page_start(page_id, i, len(pages))

        if page_id in already_done:
            log.page_skipped(page_id, "already extracted")
            skipped_pages += 1
            continue

        try:
            meta, extracted, empty = extract_page(
                page_path, page_id=page_id, logger=log
            )

            if extracted == 0:
                log.page_skipped(page_id, f"no symbols found ({empty} empty cells — likely intro/index page)")
                skipped_pages += 1
                continue

            all_meta.extend(meta)
            total_symbols += extracted
            processed     += 1
            log.page_done(page_id, extracted, empty)

            # Save after every page so progress is never lost
            with open(meta_path, "w") as f:
                json.dump(all_meta, f, indent=2)

        except Exception as e:
            log.page_error(page_id, str(e))
            error_pages += 1
            continue

    log.summary(processed, total_symbols, skipped_pages, error_pages)


# ─────────────────────────────────────────────
# Tagging UI
# ─────────────────────────────────────────────

def run_tagging_ui():
    setup_folders()
    crops_dir = OUTPUT_DIR / "crops"
    meta_path = OUTPUT_DIR / "metadata.json"

    if not meta_path.exists():
        print(f"No metadata.json found. Run --batch first.")
        return

    with open(meta_path) as f:
        records = json.load(f)

    tagged   = sum(1 for r in records if r.get("tags") and "delete" not in r["tags"])
    untagged = sum(1 for r in records if not r.get("tags"))

    print(f"\n{'='*50}")
    print(f"Event Glyph — Tagging UI")
    print(f"{'='*50}")
    print(f"Total    : {len(records)}")
    print(f"Tagged   : {tagged}")
    print(f"Remaining: {untagged}")
    print(f"\nKeys:")
    for key, label in TAXONOMY.items():
        print(f"  [{key}]  {label}")
    print(f"  [SPACE] skip  [z] undo  [q] quit & save")
    print(f"{'='*50}\n")

    history = []

    for i, record in enumerate(records):
        if record.get("tags") and "unsure" not in record["tags"]:
            continue

        img_path = crops_dir / record["file"]
        if not img_path.exists():
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        display = cv2.resize(img, (400, 400), interpolation=cv2.INTER_NEAREST)

        # Progress bar
        progress = int((i / len(records)) * 400)
        cv2.rectangle(display, (0, 0), (progress, 5), (0, 180, 90), -1)

        # Info overlay
        name = (record.get("symbol_name") or record["file"])[:40]
        cv2.putText(display, f"{i+1}/{len(records)}",
                    (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (80, 80, 80), 1)
        cv2.putText(display, name,
                    (8, 44), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (40, 40, 40), 1)
        if record.get("tags"):
            cv2.putText(display, " + ".join(record["tags"]),
                        (8, 386), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 140, 0), 1)

        cv2.imshow("Event Glyph — Tagger  [q=quit]", display)
        key = cv2.waitKey(0) & 0xFF

        if key == ord("q"):
            break
        elif key == ord("z") and history:
            prev = history.pop()
            records[prev]["tags"] = []
            print(f"  Undid: {records[prev].get('symbol_name') or records[prev]['file']}")
        elif key == ord(" "):
            continue
        elif chr(key) in TAXONOMY:
            tag = TAXONOMY[chr(key)]
            record["tags"] = ["delete"] if tag == "delete" else [tag]
            history.append(i)
            print(f"  {name} → {tag}")

    cv2.destroyAllWindows()

    # Save metadata
    with open(meta_path, "w") as f:
        json.dump(records, f, indent=2)

    # Copy tagged crops into category subfolders
    # This produces the folder structure PyTorch ImageFolder expects
    print("\nSorting crops into tagged/ subfolders...")
    copied = 0
    for record in records:
        if not record.get("tags"):
            continue
        tag = record["tags"][0]
        if tag in ("delete", "unsure"):
            continue
        src = crops_dir / record["file"]
        dst = OUTPUT_DIR / "tagged" / tag / record["file"]
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            copied += 1

    tagged_total = sum(1 for r in records if r.get("tags") and "delete" not in r["tags"])
    print(f"\nDone")
    print(f"  {tagged_total} symbols tagged")
    print(f"  {copied} crops sorted into tagged/ subfolders")
    print(f"\nLoad in PyTorch with:")
    print(f"  datasets.ImageFolder(root='{OUTPUT_DIR}/tagged', transform=transform)")


# ─────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Event Glyph — Symbol Extractor")
    group  = parser.add_mutually_exclusive_group(required=True)
    group.add_argument("--preview", action="store_true",
                       help="Test grid on a single page")
    group.add_argument("--batch",   action="store_true",
                       help="Extract symbols from all pages in a folder")
    group.add_argument("--tag",     action="store_true",
                       help="Run the tagging UI")

    parser.add_argument("--input", help="Image file (--preview) or folder (--batch)")
    args = parser.parse_args()

    if args.preview:
        if not args.input:
            print("--preview requires --input <image path>")
            sys.exit(1)
        setup_folders()
        extract_page(args.input, preview=True)

    elif args.batch:
        if not args.input:
            print("--batch requires --input <folder path>")
            sys.exit(1)
        batch_extract(args.input)

    elif args.tag:
        run_tagging_ui()

usage: ipykernel_launcher.py [-h] (--preview | --batch | --tag)
                             [--input INPUT]
ipykernel_launcher.py: error: one of the arguments --preview --batch --tag is required


SystemExit: 2

/opt/miniconda3/envs/eventGlyth/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
